# Project Assignment A - TMDB Actor Community Analysis

This notebook implements a reproducible pipeline for **Project Assignment A** in Computational Social Science.

## Central Question
Do communities of actors specialize in particular genres/themes, and who are the bridge actors across communities?

## Assignment A coverage
- Dataset source and download strategy (TMDB API)
- Implementation plan and preliminary analysis
- Dataset summary (size, rows, variables)
- Network summary (nodes, links, degree patterns, node attributes)
- Text summary (movie overviews) and network-text tie-in

## 1. Set Up Notebook Environment

Import libraries, check package availability, and set reproducibility defaults.

In [ ]:
import os
import re
import json
import time
import math
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Optional package for Louvain
try:
    import community as community_louvain  # python-louvain package
    LOUVAIN_AVAILABLE = True
except Exception:
    LOUVAIN_AVAILABLE = False

# Optional package for .env
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("load env available")
except Exception:
    pass

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Python setup ready.")
print(f"pandas={pd.__version__}, numpy={np.__version__}, networkx={nx.__version__}")
print(f"sklearn_available={SKLEARN_AVAILABLE}, louvain_available={LOUVAIN_AVAILABLE}")

Python setup ready.
pandas=3.0.2, numpy=2.4.4, networkx=3.6.1
sklearn_available=True, louvain_available=True


## 2. Define Project Constants and Configuration

Configuration for paths, API, and runtime scope. Update these settings as needed for your Assignment A timeline.

In [ ]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT_DIR / "outputs"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "api_base": "https://api.themoviedb.org/3",
    "api_key": os.getenv("TMDB_API_KEY", ""),
    "language": "en-US",
    "region": "US",
    "discover_sort_by": "popularity.desc",
    "release_date_gte": "2010-01-01",
    "release_date_lte": "2025-12-31",
    "vote_count_gte": 200,
    "max_discover_pages": 10,
    "max_cast_rank": 8,
    "sleep_between_calls_sec": 0.2,
}

print("Root:", ROOT_DIR)
print("TMDB key configured:", bool(CONFIG["api_key"]))
if not CONFIG["api_key"]:
    print("WARNING: Missing TMDB_API_KEY. Set it in your .env file to run data collection cells.")

## 3. Create Core Function Stubs

Define the main functions for API access, table construction, network building, and text analysis.

In [ ]:
def cache_path(name: str) -> Path:
    """Return a JSON cache file path in data/raw."""
    safe = re.sub(r"[^a-zA-Z0-9_\-]", "_", name)
    return RAW_DIR / f"{safe}.json"


def load_json_cache(path: Path):
    """Load JSON if present, else return None."""
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def save_json_cache(path: Path, payload) -> None:
    """Write JSON payload to cache path."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def tmdb_get(endpoint: str, params: dict | None = None, retries: int = 3, timeout: int = 20):
    """GET helper with API key, retries, and basic backoff."""
    if not CONFIG["api_key"]:
        raise RuntimeError("TMDB_API_KEY missing in environment or .env")
    url = f"{CONFIG['api_base']}/{endpoint.lstrip('/')}"
    q = dict(params or {})
    q["api_key"] = CONFIG["api_key"]

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, params=q, timeout=timeout)
            if resp.status_code == 200:
                time.sleep(CONFIG["sleep_between_calls_sec"])
                return resp.json()
            if resp.status_code in (401, 403):
                raise RuntimeError(f"Authentication error {resp.status_code}: {resp.text[:200]}")
            if resp.status_code == 429:
                wait = 1.5 * attempt
                print(f"Rate limited. Waiting {wait:.1f}s")
                time.sleep(wait)
                continue
            if 500 <= resp.status_code < 600:
                wait = 1.0 * attempt
                print(f"Server error {resp.status_code}. Retry in {wait:.1f}s")
                time.sleep(wait)
                continue
            raise RuntimeError(f"TMDB error {resp.status_code}: {resp.text[:200]}")
        except requests.RequestException as exc:
            if attempt == retries:
                raise
            wait = 1.0 * attempt
            print(f"Network error: {exc}. Retry in {wait:.1f}s")
            time.sleep(wait)


def discover_movies(max_pages: int):
    """Discover popular movies and return list of movie dicts."""
    movies = []
    for page in range(1, max_pages + 1):
        payload = tmdb_get(
            "/discover/movie",
            params={
                "page": page,
                "sort_by": CONFIG["discover_sort_by"],
                "language": CONFIG["language"],
                "region": CONFIG["region"],
                "vote_count.gte": CONFIG["vote_count_gte"],
                "release_date.gte": CONFIG["release_date_gte"],
                "release_date.lte": CONFIG["release_date_lte"],
            },
        )
        movies.extend(payload.get("results", []))
    return movies


def get_movie_details(movie_id: int):
    """Fetch full movie details including genres and overview."""
    return tmdb_get(f"/movie/{movie_id}", params={"language": CONFIG["language"]})


def get_movie_credits(movie_id: int):
    """Fetch cast credits for a movie."""
    return tmdb_get(f"/movie/{movie_id}/credits", params={"language": CONFIG["language"]})


def build_tables(movie_records: list, details_by_id: dict, credits_by_id: dict):
    """Build movies_df, actors_df, actor_movie_df from raw records."""
    movie_rows = []
    actor_rows = {}
    actor_movie_rows = []

    for m in movie_records:
        mid = m.get("id")
        d = details_by_id.get(str(mid), {})

        movie_rows.append(
            {
                "movie_id": mid,
                "title": d.get("title", m.get("title")),
                "release_date": d.get("release_date", m.get("release_date")),
                "vote_average": d.get("vote_average", m.get("vote_average")),
                "vote_count": d.get("vote_count", m.get("vote_count")),
                "popularity": d.get("popularity", m.get("popularity")),
                "overview": d.get("overview", ""),
                "genres": [g.get("name") for g in d.get("genres", []) if g.get("name")],
            }
        )

        cast = credits_by_id.get(str(mid), {}).get("cast", [])
        for c in cast:
            cast_order = c.get("order", 999)
            if cast_order > CONFIG["max_cast_rank"]:
                continue
            aid = c.get("id")
            if aid is None:
                continue
            actor_rows[aid] = {
                "actor_id": aid,
                "name": c.get("name"),
                "gender": c.get("gender"),
                "known_for_department": c.get("known_for_department"),
                "popularity": c.get("popularity"),
            }
            actor_movie_rows.append(
                {
                    "actor_id": aid,
                    "movie_id": mid,
                    "character": c.get("character"),
                    "cast_order": cast_order,
                }
            )

    movies_df = pd.DataFrame(movie_rows).drop_duplicates(subset=["movie_id"])
    actors_df = pd.DataFrame(actor_rows.values()).drop_duplicates(subset=["actor_id"])
    actor_movie_df = pd.DataFrame(actor_movie_rows).drop_duplicates(subset=["actor_id", "movie_id"])

    return movies_df, actors_df, actor_movie_df


def build_weighted_actor_graph(actor_movie_df: pd.DataFrame):
    """Create undirected weighted graph where edge weight=co-appearances."""
    pair_counter = Counter()
    for movie_id, group in actor_movie_df.groupby("movie_id"):
        actor_ids = sorted(group["actor_id"].dropna().astype(int).unique().tolist())
        for a, b in combinations(actor_ids, 2):
            pair_counter[(a, b)] += 1

    G = nx.Graph()
    for (a, b), w in pair_counter.items():
        G.add_edge(a, b, weight=int(w))
    return G


def clean_and_tokenize(text: str):
    """Simple tokenization for overviews, excluding short tokens and digits."""
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 2]
    stopwords = {
        "the", "and", "for", "with", "that", "this", "from", "have", "they", "their", "were", "will", "into",
        "about", "there", "which", "when", "where", "while", "after", "before", "through", "during", "between",
        "into", "over", "under", "than", "then", "them", "his", "her", "she", "him", "our", "you", "your",
        "who", "what", "why", "how", "not", "are", "was", "had", "has", "all", "can", "one", "two", "new",
        "life", "man", "woman", "young", "story", "find", "must", "against", "after", "time"
    }
    return [t for t in tokens if t not in stopwords]


def top_tf_terms(tokens_list: list, n: int = 10):
    """Return top terms by raw term frequency."""
    return Counter(tokens_list).most_common(n)

## 4. Implement the First Working Feature

Run a minimal data-collection slice and verify that core tables can be built.

In [ ]:
# Minimal viable run for Assignment A (small sample first).
sample_pages = min(2, CONFIG["max_discover_pages"])

if not CONFIG["api_key"]:
    print("Skipping API run: set TMDB_API_KEY in .env first.")
    sample_movies = []
else:
    sample_movies = discover_movies(sample_pages)

print(f"Discovered movies: {len(sample_movies)}")

sample_movies_cache = cache_path("sample_movies")
save_json_cache(sample_movies_cache, sample_movies)

sample_details_by_id = {}
sample_credits_by_id = {}

if sample_movies:
    for m in sample_movies:
        mid = m["id"]
        sample_details_by_id[str(mid)] = get_movie_details(mid)
        sample_credits_by_id[str(mid)] = get_movie_credits(mid)

    save_json_cache(cache_path("sample_movie_details"), sample_details_by_id)
    save_json_cache(cache_path("sample_movie_credits"), sample_credits_by_id)

movies_df, actors_df, actor_movie_df = build_tables(sample_movies, sample_details_by_id, sample_credits_by_id)
print(f"movies_df={len(movies_df)}, actors_df={len(actors_df)}, actor_movie_df={len(actor_movie_df)}")

movies_df.head(3)

## 5. Add Input Validation and Error Handling

Add explicit validation to avoid silent failures and to ensure reproducibility.

In [ ]:
def validate_config(config: dict):
    if not isinstance(config, dict):
        raise TypeError("CONFIG must be a dictionary")

    if config.get("max_discover_pages", 0) <= 0:
        raise ValueError("max_discover_pages must be > 0")

    if config.get("max_cast_rank", -1) < 0:
        raise ValueError("max_cast_rank must be >= 0")

    gte = config.get("release_date_gte", "")
    lte = config.get("release_date_lte", "")
    if gte and lte and gte > lte:
        raise ValueError("release_date_gte cannot be later than release_date_lte")


def validate_tables(movies_df: pd.DataFrame, actors_df: pd.DataFrame, actor_movie_df: pd.DataFrame):
    required_movie_cols = {"movie_id", "title", "overview", "genres"}
    required_actor_cols = {"actor_id", "name"}
    required_link_cols = {"actor_id", "movie_id"}

    if not required_movie_cols.issubset(set(movies_df.columns)):
        raise ValueError("movies_df missing required columns")
    if not required_actor_cols.issubset(set(actors_df.columns)):
        raise ValueError("actors_df missing required columns")
    if not required_link_cols.issubset(set(actor_movie_df.columns)):
        raise ValueError("actor_movie_df missing required columns")

    if actor_movie_df[["actor_id", "movie_id"]].isna().any().any():
        raise ValueError("actor_movie_df contains null actor_id/movie_id")

validate_config(CONFIG)
print("Config validation passed.")
if len(movies_df) and len(actors_df) and len(actor_movie_df):
    validate_tables(movies_df, actors_df, actor_movie_df)
    print("Table validation passed for sample run.")

## 6. Write Quick Unit Tests in Notebook

Lightweight assert-based checks for normal and edge/failure paths.

In [ ]:
# Tokenizer tests
assert clean_and_tokenize("Hello, WORLD! 123") == ["hello", "world"]
assert clean_and_tokenize(None) == []

# Graph builder tests
_tmp = pd.DataFrame(
    [
        {"actor_id": 1, "movie_id": 10},
        {"actor_id": 2, "movie_id": 10},
        {"actor_id": 2, "movie_id": 20},
        {"actor_id": 3, "movie_id": 20},
    ]
)
G_test = build_weighted_actor_graph(_tmp)
assert G_test.number_of_nodes() == 3
assert G_test.number_of_edges() == 2
assert G_test[1][2]["weight"] == 1
assert G_test[2][3]["weight"] == 1

# Failure path test
try:
    validate_config({"max_discover_pages": 0, "max_cast_rank": 2, "release_date_gte": "2020-01-01", "release_date_lte": "2021-01-01"})
    raise AssertionError("Expected ValueError for invalid max_discover_pages")
except ValueError:
    pass

print("Quick tests passed.")

## 7. Run an End-to-End Execution Cell

Run the full Assignment A preliminary pipeline on configured scope.

In [ ]:
if not CONFIG["api_key"]:
    print("TMDB_API_KEY is missing. Configure .env to execute end-to-end pipeline.")
else:
    # 1) Collect movies
    movies = discover_movies(CONFIG["max_discover_pages"])
    save_json_cache(cache_path("discover_movies_full"), movies)
    print(f"Collected movies: {len(movies)}")

    # 2) Collect movie details + credits
    details_by_id = {}
    credits_by_id = {}
    for m in movies:
        mid = m["id"]
        details_by_id[str(mid)] = get_movie_details(mid)
        credits_by_id[str(mid)] = get_movie_credits(mid)

    save_json_cache(cache_path("movie_details_full"), details_by_id)
    save_json_cache(cache_path("movie_credits_full"), credits_by_id)

    # 3) Build canonical tables
    movies_df, actors_df, actor_movie_df = build_tables(movies, details_by_id, credits_by_id)
    validate_tables(movies_df, actors_df, actor_movie_df)

    movies_df.to_parquet(PROCESSED_DIR / "movies.parquet", index=False)
    actors_df.to_parquet(PROCESSED_DIR / "actors.parquet", index=False)
    actor_movie_df.to_parquet(PROCESSED_DIR / "actor_movie.parquet", index=False)

    print("Rows:")
    print(f"movies_df={len(movies_df):,}, actors_df={len(actors_df):,}, actor_movie_df={len(actor_movie_df):,}")

    # 4) Build weighted actor network
    G = build_weighted_actor_graph(actor_movie_df)

    # Add actor attributes
    actor_map = actors_df.set_index("actor_id").to_dict(orient="index")
    for node in G.nodes:
        if node in actor_map:
            nx.set_node_attributes(G, {node: actor_map[node]})

    # 5) Basic network stats
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    density = nx.density(G) if n_nodes > 1 else 0.0
    components = list(nx.connected_components(G)) if n_nodes > 0 else []
    n_components = len(components)
    isolates = list(nx.isolates(G)) if n_nodes > 0 else []
    gcc_nodes = max(components, key=len) if components else set()
    G_gcc = G.subgraph(gcc_nodes).copy() if gcc_nodes else nx.Graph()

    print("\nNetwork stats:")
    print(f"nodes={n_nodes:,}, edges={n_edges:,}, density={density:.6f}")
    print(f"components={n_components}, isolates={len(isolates):,}, GCC size={len(gcc_nodes):,}")

    # 6) Degree and strength distributions
    deg = np.array([d for _, d in G.degree()], dtype=float)
    strength = np.array([s for _, s in G.degree(weight="weight")], dtype=float)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(deg, bins=30, color="#1f77b4", alpha=0.8)
    ax[0].set_title("Degree Distribution")
    ax[0].set_xlabel("Degree")
    ax[0].set_ylabel("Count")

    ax[1].hist(strength, bins=30, color="#ff7f0e", alpha=0.8)
    ax[1].set_title("Strength Distribution")
    ax[1].set_xlabel("Strength")
    ax[1].set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "fig_degree_strength_distributions.png", dpi=160)
    plt.show()

    # 7) Centrality metrics on GCC
    centrality_rows = []
    if G_gcc.number_of_nodes() > 2:
        degree_cent = nx.degree_centrality(G_gcc)
        close_cent = nx.closeness_centrality(G_gcc)

        try:
            eig_cent = nx.eigenvector_centrality(G_gcc, max_iter=1000)
        except Exception:
            eig_cent = {n: np.nan for n in G_gcc.nodes()}

        try:
            bet_cent = nx.betweenness_centrality(G_gcc, weight="weight")
        except Exception:
            bet_cent = {n: np.nan for n in G_gcc.nodes()}

        for n in G_gcc.nodes():
            row = {
                "actor_id": n,
                "degree": G_gcc.degree(n),
                "strength": G_gcc.degree(n, weight="weight"),
                "degree_centrality": degree_cent.get(n, np.nan),
                "closeness_centrality": close_cent.get(n, np.nan),
                "eigenvector_centrality": eig_cent.get(n, np.nan),
                "betweenness_centrality": bet_cent.get(n, np.nan),
                "name": G.nodes[n].get("name", "Unknown"),
            }
            centrality_rows.append(row)

    centrality_df = pd.DataFrame(centrality_rows).sort_values("degree", ascending=False)
    centrality_df.to_csv(OUTPUTS_DIR / "table_top_central_actors.csv", index=False)
    print("\nTop actors by degree:")
    display(centrality_df.head(10))

    # 8) Community detection
    if G_gcc.number_of_nodes() > 0 and LOUVAIN_AVAILABLE:
        partition = community_louvain.best_partition(G_gcc, weight="weight", random_state=RANDOM_SEED)
        modularity = community_louvain.modularity(partition, G_gcc, weight="weight")
        partition_series = pd.Series(partition, name="community")
        community_sizes = partition_series.value_counts().rename_axis("community").reset_index(name="size")
        print(f"\nLouvain communities={len(community_sizes)}, modularity={modularity:.4f}")
    else:
        partition = {n: 0 for n in G_gcc.nodes()}
        modularity = np.nan
        community_sizes = pd.DataFrame({"community": [0], "size": [G_gcc.number_of_nodes()]})
        print("\nLouvain unavailable or empty graph; using single fallback community.")

    community_sizes.to_csv(OUTPUTS_DIR / "table_community_sizes.csv", index=False)
    display(community_sizes.head(15))

    plt.figure(figsize=(7, 4))
    plt.bar(community_sizes["community"].astype(str), community_sizes["size"], color="#2ca02c")
    plt.title("Community Sizes")
    plt.xlabel("Community")
    plt.ylabel("Number of actors")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "fig_community_sizes.png", dpi=160)
    plt.show()

    # 9) Tie text to communities
    if len(partition) > 0:
        actor_movie_with_comm = actor_movie_df.copy()
        actor_movie_with_comm["community"] = actor_movie_with_comm["actor_id"].map(partition)
        actor_movie_with_comm = actor_movie_with_comm.dropna(subset=["community"])
        actor_movie_with_comm["community"] = actor_movie_with_comm["community"].astype(int)

        movie_text_df = movies_df[["movie_id", "overview"]].copy()
        joined = actor_movie_with_comm.merge(movie_text_df, on="movie_id", how="left")
        joined["overview"] = joined["overview"].fillna("")

        # Aggregate per community
        docs = (
            joined.groupby("community")["overview"]
            .apply(lambda s: " ".join(s.tolist()))
            .rename("doc")
            .reset_index()
        )

        docs["tokens"] = docs["doc"].apply(clean_and_tokenize)
        docs["n_tokens"] = docs["tokens"].apply(len)

        tf_rows = []
        for _, r in docs.iterrows():
            top_terms = top_tf_terms(r["tokens"], n=10)
            for term, freq in top_terms:
                tf_rows.append({"community": r["community"], "term": term, "tf": freq})
        tf_df = pd.DataFrame(tf_rows)
        tf_df.to_csv(OUTPUTS_DIR / "table_tf_terms.csv", index=False)

        # TF-IDF
        if SKLEARN_AVAILABLE and len(docs) > 1:
            corpus = docs["tokens"].apply(lambda toks: " ".join(toks)).tolist()
            vectorizer = TfidfVectorizer(max_features=4000)
            X = vectorizer.fit_transform(corpus)
            terms = np.array(vectorizer.get_feature_names_out())

            tfidf_rows = []
            for i, comm in enumerate(docs["community"].tolist()):
                row = X[i].toarray().ravel()
                if row.sum() == 0:
                    continue
                top_idx = np.argsort(row)[-10:][::-1]
                for idx in top_idx:
                    tfidf_rows.append({"community": comm, "term": terms[idx], "tfidf": float(row[idx])})

            tfidf_df = pd.DataFrame(tfidf_rows)
            tfidf_df.to_csv(OUTPUTS_DIR / "table_tfidf_terms.csv", index=False)
            print("\nTop TF-IDF terms by community:")
            display(tfidf_df.groupby("community").head(10))
        else:
            print("TF-IDF skipped: sklearn missing or only one community available.")
    else:
        print("No partition available for text/community tie-in.")

## 8. Log Outputs and Save Artifacts

Produce slide-ready summary tables and files for Project Assignment A.

In [ ]:
# Structured metadata and Assignment A checklist outputs
run_metadata = {
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "config": CONFIG,
    "artifacts_dir": str(OUTPUTS_DIR),
    "data_dir": str(DATA_DIR),
}

if "movies_df" in globals() and isinstance(movies_df, pd.DataFrame):
    run_metadata["movies_rows"] = int(len(movies_df))
if "actors_df" in globals() and isinstance(actors_df, pd.DataFrame):
    run_metadata["actors_rows"] = int(len(actors_df))
if "actor_movie_df" in globals() and isinstance(actor_movie_df, pd.DataFrame):
    run_metadata["actor_movie_rows"] = int(len(actor_movie_df))
if "G" in globals():
    run_metadata["network_nodes"] = int(G.number_of_nodes())
    run_metadata["network_edges"] = int(G.number_of_edges())

with open(OUTPUTS_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

print("Saved run metadata:")
print(json.dumps(run_metadata, indent=2))

print("\nAssignment A slide checklist:")
print("1) Idea and why interesting: included in notebook intro markdown.")
print("2) Dataset and download method: API section + collection cells.")
print("3) Implementation plan: section flow and function architecture.")
print("4) Preliminary data analysis: rows/variables and saved summaries.")
print("5) Network definition + stats: graph construction and diagnostics.")
print("6) Text analysis + network tie-in: community documents + TF/TF-IDF.")

## How Network and Text Are Tied Together

The integration logic is:
1. Build actor communities from the weighted co-appearance network using Louvain.
2. Map each actor to a community label.
3. Join actor-community labels with the actor-movie table.
4. Collect movie overviews associated with each community.
5. Compute TF and TF-IDF per community-level document.

This makes it possible to compare **network structure** (community memberships and bridge actors) with **community-specific language/themes** in movie plots.

## Preliminary Conclusions, Limitations, and Next Steps

### Preliminary conclusions
- The actor collaboration network can be represented as a weighted graph where repeated co-appearances strengthen ties.
- Community detection provides candidate actor clusters that may correspond to thematic production ecosystems.
- TF-IDF helps identify distinguishing terms for each cluster, improving interpretability over raw term frequency.

### Limitations
- TMDB popularity and vote thresholds may bias the sample toward mainstream productions.
- Cast rank filtering improves signal but may omit relevant bridge actors.
- Overview text is short/noisy compared to full scripts or subtitles.

### Next steps toward Project Assignment B
- Expand temporal coverage and compare communities across time windows.
- Add robustness checks for community stability.
- Convert outputs into a narrative website section with non-technical explanations and downloadable data assets.